# 03 — The Bloch Sphere

Turn every pure qubit into a single point on a sphere — the compass we navigate by for the rest of the course.

**Prerequisites:** `01/01_amplitudes_and_superposition` (the two angles of `qubit_state`), `01/02_why_amplitudes_are_complex` (the relative phase $\phi$; global phase is invisible).

**What you'll be able to do**
- Place $|0\rangle$, $|1\rangle$, $|+\rangle$, $|-\rangle$ and $|{+i}\rangle$ on the Bloch sphere by eye.
- Compute a pure qubit's Bloch vector $(x, y, z)$ *by hand* from its two amplitudes, and read it as latitude and longitude.
- Explain two features of the map: a global phase does not move the point, and states as different as $|0\rangle$ and $|1\rangle$ (*orthogonal* states) sit at antipodes.

In `01/01` you wrote a qubit as two complex amplitudes and met `qubit_state`, which builds one from two angles $(\theta, \phi)$. In `01/02` you learned what the phase $\phi$ does. Those angles carry a beautiful meaning: they place the qubit on the surface of a sphere. From here on, one picture holds every pure state at once.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.colors import COLORS
from qot_course.quantum.states import KET_0, KET_1, KET_PLUS, KET_MINUS, qubit_state, bloch_vector

viz.use_course_style()

## 1. A sphere of states

Here is the central picture. Every **pure** qubit corresponds to exactly one point on the surface of a unit sphere — the **Bloch sphere**:

- the **north pole** is $|0\rangle$ and the **south pole** is $|1\rangle$ (the classical states),
- the **equator** holds the equal superpositions, like $|+\rangle$,
- every other point is some tilted, phase-shifted blend.

Let's drop our first three states onto it (the plotting helper `viz.plot_bloch` draws the sphere; how it finds the point is what section 2 makes you compute yourself).

In [ ]:
viz.plot_bloch(KET_0, title="|0>  (north pole)")
viz.plot_bloch(KET_1, title="|1>  (south pole)")
viz.plot_bloch(KET_PLUS, title="|+>  (equator)")
plt.show()

**Read the figures.** $|0\rangle$ points straight up and $|1\rangle$ straight down — the two classical states sit at opposite poles, as far apart as the sphere allows. $|+\rangle$ lies flat on the equator, along the $x$-axis, exactly between them: an equal superposition is, on the sphere, the halfway point. Distance on this sphere is about to mean something physical, and you are already reading it.

## 2. Two angles, one point — by hand

The two angles you passed to `qubit_state` in `01/01` are exactly this sphere's coordinates:

$$ |\psi\rangle = \cos(\theta/2)\,|0\rangle + e^{i\phi}\sin(\theta/2)\,|1\rangle . $$

- $\theta$ (the polar angle, $0$ at the north pole, $\pi$ at the south pole) sets the **latitude** — how much $|1\rangle$ is mixed in.
- $\phi$ (the azimuthal angle) sets the **longitude** — the relative phase of `01/02`, i.e. where around the equator the state points.

A point at polar angle $\theta$ (down from the north pole) and azimuth $\phi$ (around the $z$-axis) on a unit sphere has height $z=\cos\theta$, and its shadow in the $x$–$y$ plane has length $\sin\theta$ and points at longitude $\phi$ — the spherical coordinates

$$ (x, y, z) = (\sin\theta\cos\phi,\ \sin\theta\sin\phi,\ \cos\theta) . $$

The same three numbers can be read straight off the amplitudes $a_0 = \cos(\theta/2)$ and $a_1 = e^{i\phi}\sin(\theta/2)$. One new symbol: $\bar a$ is the **complex conjugate** of $a$ (`np.conj`) — the arrow of `01/02` reflected across the real axis, $\overline{r e^{i\varphi}} = r e^{-i\varphi}$. So a product $\bar a_0 a_1$ keeps the two sizes and *subtracts* the phases: it isolates the relative phase. Here $\bar a_0 a_1 = \cos(\theta/2)\sin(\theta/2)\,e^{i\phi} = \tfrac12\sin\theta\,e^{i\phi}$ (half-angle identity $2\cos(\theta/2)\sin(\theta/2)=\sin\theta$), and $|a_0|^2-|a_1|^2 = \cos^2(\theta/2)-\sin^2(\theta/2)=\cos\theta$. Hence:

$$ z = |a_0|^2 - |a_1|^2, \qquad x = 2\,\mathrm{Re}(\bar a_0 a_1), \qquad y = 2\,\mathrm{Im}(\bar a_0 a_1) . $$

So $z$ is the *weight imbalance* between $|0\rangle$ and $|1\rangle$, and $(x, y)$ is the *cross term* $\bar a_0 a_1$ — the object that carries the relative phase. Let's compute both versions for the tilted state, then compare with the course helper `bloch_vector`.

In [ ]:
theta, phi = 2 * np.pi / 3, np.pi / 4          # 120 deg down, 45 deg around
psi = qubit_state(theta=theta, phi=phi)
a0, a1 = psi

# by hand, version 1: spherical coordinates of (theta, phi)
r_angles = np.array([np.sin(theta) * np.cos(phi), np.sin(theta) * np.sin(phi), np.cos(theta)])
# by hand, version 2: straight from the amplitudes
cross = np.conj(a0) * a1                       # conj flips the sign of the phase: conj(r e^{i phi}) = r e^{-i phi}
print("a0, conj(a0):", np.round(a0, 4), np.round(np.conj(a0), 4), "   a1, conj(a1):", np.round(a1, 4), np.round(np.conj(a1), 4))
r_amps = np.array([2 * cross.real, 2 * cross.imag, abs(a0) ** 2 - abs(a1) ** 2])
# the course helper
r_lib = bloch_vector(psi)

print("from (theta, phi):", np.round(r_angles, 4))
print("from amplitudes  :", np.round(r_amps, 4))
print("bloch_vector     :", np.round(r_lib, 4))
print("all three agree  :", np.allclose(r_angles, r_amps) and np.allclose(r_amps, r_lib))
print("length           :", round(float(np.linalg.norm(r_amps)), 6), " (pure states sit on the surface)")

viz.plot_bloch(psi, title="|psi>  (a tilted state)")
plt.show()

**Read the figure and the output.** Three routes, one point: $(x, y, z) \approx (0.612, 0.612, -0.5)$. The tilted state points down-and-around: its latitude comes from $\theta = 120^\circ$ (past the equator, so $z<0$ — more $|1\rangle$ than $|0\rangle$, matching the weights $0.25/0.75$ from `01/01`) and its longitude from $\phi = 45^\circ$ (the phase swings it off the $x$-axis into the $x$–$y$ plane, $x = y$). Its Bloch vector has length exactly $1$ — every pure state lives on the **surface**. Hold that thought: in `01/12`–`01/13` the states *inside* the ball will turn out to be the genuinely mixed ones.

(The helper `bloch_vector` computes the same three numbers as three *expectation values*, $\langle X\rangle, \langle Y\rangle, \langle Z\rangle$ — vocabulary you earn in `01/07` and `01/10`. Today the amplitude formula is all you need.)

## 3. Two things the map gets right

**Global phase does not move the point.** `01/02` showed that $e^{i\alpha}|\psi\rangle$ is physically the same state as $|\psi\rangle$. In the formulas above, $a_0 \to e^{i\alpha}a_0$ and $a_1 \to e^{i\alpha}a_1$ leave $|a_0|^2$, $|a_1|^2$ and $\bar a_0 a_1$ untouched — so the sphere *automatically* forgets the global phase. That is why a state with two complex amplitudes (four real numbers) can live on a two-dimensional surface: normalisation removes one number, global phase removes another.

**Orthogonal states are antipodes.** We call two states *orthogonal* when they are as different as $|0\rangle$ and $|1\rangle$ — `01/02`'s phrase; `01/05` gives it the algebraic test $\bar a_0 b_0 + \bar a_1 b_1 = 0$, which you can already try in exercise 3. The sphere puts $|0\rangle$ and $|1\rangle$ at opposite poles, $180^\circ$ apart — yet as amplitude pairs, $(1,0)$ and $(0,1)$ are perpendicular axes, a quarter-turn. The sphere stretches that quarter-turn into a half-turn; that doubling is the $\theta/2$ in the formula. The same holds for $|+\rangle$ and $|-\rangle$ (which `01/02` showed to be as different as $|0\rangle$ and $|1\rangle$): opposite points on the equator. Let's check both claims numerically.

In [ ]:
alpha = np.pi / 3
print("|psi>          :", np.round(bloch_vector(psi), 4))
print("e^{i alpha}|psi>:", np.round(bloch_vector(np.exp(1j * alpha) * psi), 4), " (global phase: same point)")
print()
for name_a, sa, name_b, sb in [("|0>", KET_0, "|1>", KET_1), ("|+>", KET_PLUS, "|->", KET_MINUS)]:
    ra, rb = bloch_vector(sa), bloch_vector(sb)
    print(f"{name_a} at {np.round(ra, 3)}   {name_b} at {np.round(rb, 3)}   antipodal: {np.allclose(ra, -rb)}")

**Read the output.** Multiplying the tilted state by $e^{i\pi/3}$ leaves its Bloch vector unchanged to every printed digit — the map is blind to global phase, as it should be. $|0\rangle/|1\rangle$ sit at $(0,0,\pm1)$ and $|+\rangle/|-\rangle$ at $(\pm1,0,0)$: each pair is antipodal. From now on, "opposite on the sphere" is your visual for "as different as $|0\rangle$ and $|1\rangle$" (`01/05` makes this an inner product, `01/04` a measurement).

## 4. Walking the equator

The equator is where the relative phase lives: every state on it has weights $0.5/0.5$, and only the longitude $\phi$ distinguishes them. Let's walk around it with `qubit_state(theta=pi/2, phi)` and mark four states — the three you already know, $|+\rangle$ at $\phi=0$, $|{+i}\rangle$ at $\phi=\pi/2$ (from `01/02`, section 4) and $|-\rangle$ at $\phi=\pi$, plus the mirror of $|{+i}\rangle$, $|{-i}\rangle = \tfrac{1}{\sqrt2}\big(|0\rangle - i\,|1\rangle\big)$ at $\phi = 3\pi/2$ (a $-i$ is a $270^\circ$ arrow, per `01/02`) — seen from above the north pole.

In [ ]:
phis = np.linspace(0, 2 * np.pi, 200)
ring = np.array([bloch_vector(qubit_state(theta=np.pi / 2, phi=f)) for f in phis])   # shape (200, 3)

marks = [(0.0, r"$|+\rangle$", COLORS["source"]), (np.pi / 2, r"$|{+i}\rangle$", COLORS["quantum"]),
         (np.pi, r"$|-\rangle$", COLORS["target"]), (3 * np.pi / 2, r"$|{-i}\rangle$", COLORS["highlight"])]

fig, ax = plt.subplots(figsize=(5.2, 5.2))
ax.plot(ring[:, 0], ring[:, 1], color=COLORS["grid"], lw=1.5, label=r"equator ($\theta=\pi/2$)")
for f, lab, col in marks:
    x, y, z = bloch_vector(qubit_state(theta=np.pi / 2, phi=f))
    ax.scatter([x], [y], color=col, s=70, zorder=5)
    ax.text(1.15 * x, 1.15 * y, lab, color=col, ha="center", va="center")
ax.axhline(0, color=COLORS["grid"], lw=0.8); ax.axvline(0, color=COLORS["grid"], lw=0.8)
ax.set_xlim(-1.35, 1.35); ax.set_ylim(-1.35, 1.35); ax.set_aspect("equal")
ax.set_xlabel("x"); ax.set_ylabel("y")
ax.set_title("The equator seen from the north pole: longitude = relative phase", pad=12)
ax.legend(loc="lower left")
plt.show()

print("z on the whole equator is 0:", np.allclose(ring[:, 2], 0.0), "  (weights 0.5/0.5 everywhere)")

**Read the figure.** Seen from above, the equator is the unit circle in the $x$–$y$ plane and $z=0$ all the way round — every equatorial state has weights $0.5/0.5$. The four marked states differ *only* by longitude: $|+\rangle$ (periwinkle) at $0^\circ$, $|{+i}\rangle$ (sky blue) at $90^\circ$, $|-\rangle$ (amber) at $180^\circ$, and $|{-i}\rangle$ (rose) at $270^\circ$. Compare with the complex-plane figure of `01/02`: the arrow angle of the $|1\rangle$ amplitude *is* the longitude here — the sphere wraps that phase circle around the equator. And the two states `01/02` used to argue for complex amplitudes, $|+\rangle$ and $|{+i}\rangle$, sit a quarter-turn apart: neither is a pole, neither is the other's antipode.

## Your turn

1. **Find the poles' angles.** Which $\theta$ gives $|0\rangle$? Which gives $|1\rangle$? Confirm with `qubit_state` and `bloch_vector`, and explain why $\phi$ does not matter at the poles (look at the amplitude formulas for $x$ and $y$).
2. **Predict a vector.** Pick your own $(\theta, \phi)$, predict the Bloch $(x, y, z)$ *on paper* from $(\sin\theta\cos\phi, \sin\theta\sin\phi, \cos\theta)$, then check with `bloch_vector`. How close were you?
3. **The antipode of anything.** For your state from exercise 2, write down the state at the antipodal point (hint: $\theta \to \pi - \theta$, $\phi \to \phi + \pi$). Build it with `qubit_state`, confirm `bloch_vector` gives the negated vector, and check that the two states' amplitude vectors satisfy $\bar a_0 b_0 + \bar a_1 b_1 = 0$ (the algebraic face of "orthogonal", made official in `01/05`).
4. **Harder: latitude and weights.** Show on paper that the weight of $|1\rangle$ is $(1 - z)/2$ for every pure state, then verify it numerically on 1000 random $(\theta, \phi)$ (use `rng = np.random.default_rng(0)`) with `np.allclose`. What does this say about how far a state must be from the north pole for its $|1\rangle$ weight to reach $0.9$?

## What you built

- You placed $|0\rangle$, $|1\rangle$, $|+\rangle$, $|-\rangle$, $|{\pm i}\rangle$ on the Bloch sphere — poles and equator — by eye.
- You computed a Bloch vector three ways — from $(\theta,\phi)$, from the amplitudes ($z = |a_0|^2-|a_1|^2$, $x+iy = 2\bar a_0 a_1$), and with `bloch_vector` — and saw they agree; every pure state has length $1$.
- You checked that the map forgets global phase and puts orthogonal states at antipodes — the $\theta/2$ in the formula.
- You walked the equator and saw longitude = relative phase, wrapping `01/02`'s phase circle around the sphere.

You now hold a single mental image for the entire space of pure qubits — a compass you will reach for again and again. That is a lot of territory captured in one picture.

**Dictionary so far** (the table from `01/02`, plus the row you earned today):

| Classical | Quantum |
|-----------|---------|
| bit ($0$ or $1$) | qubit $a_0\,|0\rangle + a_1\,|1\rangle$ (two complex amplitudes) |
| probabilities only add (non-negative) | amplitudes add *and cancel* — interference; a relative phase $e^{i\varphi}$ is physical, a global phase is not |
| a bit's state space: two isolated points | a pure qubit's state space: the Bloch **sphere** (poles = the classical bit; equator = equal superpositions) |
| probability vector `p` | density matrix $\rho$ (diagonal in a shared basis $\Rightarrow$ classical) |
| marginal | partial trace |
| transport plan | quantum coupling |

(The last three rows are still forward-looking — earned in `01/12`, `01/16` and `04/04`.)

## What's next

We can *place* a state, but we still cannot *read* one: a measurement only ever returns a bare $0$ or $1$. In `01/04_measurement_and_born_rule` we turn amplitudes into probabilities — the weights $|a|^2$ become the Born rule, and $z$ becomes the bias of the coin you actually flip — and discover what a measurement can, and cannot, see.

## References

- M. A. Nielsen & I. L. Chuang, *Quantum Computation and Quantum Information*, §1.2 (the Bloch sphere), Cambridge University Press (2010). doi:10.1017/CBO9780511976667
- F. Bloch, "Nuclear induction", *Physical Review* 70, 460–474 (1946) — historical origin of the picture (spin-1/2 magnetisation). doi:10.1103/PhysRev.70.460
- IBM Quantum Learning (formerly the Qiskit textbook), "Representing Qubit States" (Bloch sphere), https://learning.quantum.ibm.com/

**Previous:** `notebooks/01_QuantumFoundations/02_why_amplitudes_are_complex.ipynb` · **Next:** `notebooks/01_QuantumFoundations/04_measurement_and_born_rule.ipynb`